In [1]:
# ---- MBE + GAPT ---- 
# 1. Modified GPT (with MBE calculation)
from src.gapt import GPTConfig, GPT
import torch 

config = GPTConfig(
    n_layer=4,
    n_head=4,
    n_embd=128,
)
    
model = GPT(config)
# model = model.to("cuda")
# model = torch.compile(model)

In [2]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0
        
        buf = tokens[pos : pos + sequence_length + 1]
        inputs = buf[None, :-1].to(device=device, dtype=torch.int32, non_blocking=True) # no sync on host side;
        targets = buf[None, 1:].to(device=device, dtype=torch.int64, non_blocking=True) 
        pos += sequence_length
        yield inputs, targets

data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))
train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=16, device="cpu")

In [3]:
import time
from src.mbe import patch_mbe
from src.gradtracker import GradientTracker

g = GradientTracker(model)

# --- forward propagation ---
input, target = next(train_loader)

model.enable_timing = True
output = model(input, target, attn_blocksize=256, patch_size=8)
model.enable_timing = False

# --- backward propagation --- 
# Question 1. if I backward pass on added MBE loss, will it has the same speed & memory usage? 
g.backward_with_tracking({"mbe_0": output["mbe_0"]}, retain_graph=True)
g.backward_with_tracking({"mbe_1": output["mbe_1"]}, retain_graph=True)
g.backward_with_tracking({"mbe_2": output["mbe_2"]}, retain_graph=True)
g.backward_with_tracking({"mbe": output["mbe_0"] + output["mbe_1"] + output["mbe_2"]}, retain_graph=True)
g.backward_with_tracking({"ce": output["entropy"]})


⏱️  Forward Pass Timing Breakdown (ms)
Setup (mask + embed):         2.73 ms  (  1.7%)
Encoder Forward:            147.64 ms  ( 92.3%)
Encoder MBE:                  0.23 ms  (  0.1%)
Decoder Forward:              6.42 ms  (  4.0%)
Decoder MBE:                  0.09 ms  (  0.1%)
Output (head + loss):         2.84 ms  (  1.8%)
------------------------------------------------------------
TOTAL:                      159.96 ms



In [10]:
# Toy training loop 
# -------------------------------------------------------------
attn_blocksize = 1792 
patch_size = 16 
no_reg = False 
use_gapt = True
log_grad_info = False
mbe_weight = 1.0

from src.utils import compute_loss
from src.gapt import GatedPhaseTransition
from src.gradtracker import GradientTracker
from torch.optim import Adam

grad_tracker = GradientTracker(model)
gapt = GatedPhaseTransition()
iterations = 100 
optimizer = Adam(model.parameters(), lr=0.001)

inputs, targets = next(train_loader)

for i in range(iterations): 
    # inputs, targets = next(train_loader)
    loss_dict = model.forward(inputs, targets, attn_blocksize, patch_size)
    compute_loss(loss_dict)  

    # --- aggregate loss ---
    loss_dict = {
        "entropy": loss_dict["entropy"],
        "mbe": sum(v for k, v in loss_dict.items() if k.startswith("mbe_"))
    }
    if no_reg: 
        loss_dict = {"entropy": loss_dict["entropy"]}
    elif use_gapt:
        loss = gapt.step(loss_dict["entropy"], loss_dict["mbe"], verbose=False)
        loss_name = "entropy" if gapt.phi == 1 else "mbe"
        loss_dict = {loss_name: loss}
    else:
        loss_dict = {"combined": loss_dict["entropy"] + mbe_weight * loss_dict["mbe"]}

    # --- backward ---
    if log_grad_info: 
        grad_tracker.backward_with_tracking(loss_dict)
    else: 
        grad_tracker.backward(loss_dict)

    optimizer.step()
    optimizer.zero_grad()    

    print(f" - step {i}/{iterations} " + "".join([f" {k}={v:.2f}" for k, v in loss_dict.items()]))

 - step 0/100  entropy=9.68
 - step 1/100  entropy=9.28
 - step 2/100  entropy=8.92
 - step 3/100  entropy=8.56
 - step 4/100  entropy=8.23
 - step 5/100  entropy=7.94
 - step 6/100  entropy=7.67
 - step 7/100  entropy=7.43
 - step 8/100  entropy=7.21
 - step 9/100  entropy=7.01
 - step 10/100  entropy=6.76
 - step 11/100  entropy=6.53
 - step 12/100  entropy=6.33
 - step 13/100  entropy=6.06
 - step 14/100  entropy=5.87
 - step 15/100  entropy=5.68
 - step 16/100  entropy=5.48
 - step 17/100  entropy=5.27
 - step 18/100  entropy=5.06
 - step 19/100  entropy=4.85
 - step 20/100  entropy=4.67
 - step 21/100  entropy=4.50
 - step 22/100  entropy=4.30
 - step 23/100  entropy=4.12
 - step 24/100  entropy=3.95
 - step 25/100  entropy=3.78
 - step 26/100  entropy=3.61
 - step 27/100  entropy=3.44
 - step 28/100  entropy=3.29
 - step 29/100  entropy=3.14
 - step 30/100  entropy=3.01
 - step 31/100  entropy=2.88
 - step 32/100  entropy=2.75
 - step 33/100  entropy=2.65
 - step 34/100  entropy=